# Lite-DPAS V5: 16×16 Spatially Preserved Latent

이 실험은 V3/V4의 8×8 bottleneck이 CSI 세부 정보를 너무 일찍 줄이는지 확인하기 위한 **구조 변경 실험**입니다. 증류 없이, 같은 feedback dimension에서 16×16 공간 해상도를 유지하며 8채널로 처리한 뒤 4채널 latent로 투영하고 delay 축 depthwise kernel을 15로 넓힙니다.

- Encoder: 32×32 → 16×16까지만 다운샘플링한 뒤 spectral/delay refinement
- Latent: 16×16×4 → 기존과 같은 `ENCODED_DIM` 실수 codeword
- Decoder: codeword → 16×16×8 → 한 번의 PixelShuffle로 32×32 복원
- 학습: validation과 정렬된 sample-wise NMSE + plateau 기반 LR 감소
- 증류는 사용하지 않으며 test split은 열지 않습니다.

이 설계는 성능 개선을 보장하지 않는 가설입니다. NMSE와 함께 실제 지연 및 파라미터/MAC을 비교하세요.


In [ ]:
from pathlib import Path
import json, sys, time, shutil
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# Colab에서는 프로젝트 폴더를 Drive/MyDrive/CSI_SLLM_RESEARCH에 두세요.
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/CSI_SLLM_RESEARCH')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / 'src').is_dir() and (PROJECT_ROOT / 'CSI_SLLM_RESEARCH' / 'src').is_dir():
        PROJECT_ROOT = PROJECT_ROOT / 'CSI_SLLM_RESEARCH'
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('CSI_SLLM_RESEARCH 프로젝트 폴더를 찾지 못했습니다. PROJECT_ROOT를 직접 지정하세요.')
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / 'Data'
if IN_COLAB:
    # MAT 파일은 세션 로컬에서 읽고, checkpoint와 로그는 Drive의 PROJECT_ROOT에 저장합니다.
    LOCAL_DATA_DIR = Path('/content/cost2100_indoor')
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name in ['DATA_Htrainin.mat', 'DATA_Hvalin.mat']:
        src, dst = DATA_DIR / name, LOCAL_DATA_DIR / name
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)
    DATA_DIR = LOCAL_DATA_DIR
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

In [ ]:
# 실험 설정: 첫 실행은 1 epoch로 확인한 다음 전체 학습으로 바꾸세요.
COMPRESSION_RATIO = 4       # 지원값: 4, 16, 32, 64
EPOCHS = 1000               # 예비 확인은 1, 본 학습은 1000
BATCH_SIZE = 200            # GPU 메모리 부족 시 64 또는 32
SEED = 42
RUN_TAG = 'v5_spatial_preserve_01'      # 기존 v2_fresh_01 결과 보존; 재실험은 새 태그 사용
SAVE_EVERY = 100            # epoch 주기 checkpoint 저장
RESUME_FROM = None          # 예: Path('/content/drive/MyDrive/.../checkpoint_epoch_0100.pt')
RUN_TRAINING = False        # 학습을 시작하려면 True로 변경
PROFILE_SAMPLES = 512      # validation 프로파일에 사용할 표본 수

assert COMPRESSION_RATIO in (4, 16, 32, 64)
ENCODED_DIM = 2048 // COMPRESSION_RATIO
required = ['DATA_Htrainin.mat', 'DATA_Hvalin.mat']
missing = [name for name in required if not (DATA_DIR / name).is_file()]
assert not missing, f'Data 폴더에 파일이 없습니다: {missing}'
print(f'CR=1/{COMPRESSION_RATIO} | latent={ENCODED_DIM} real values | epochs={EPOCHS} | batch={BATCH_SIZE}')
print('Train/validation 파일 확인 완료. Test 파일은 이 노트북에서 열지 않습니다.')

LOSS_MODE = 'nmse'
SCHEDULER_PATIENCE = 25


## 1. V5 모델 동작 확인

무작위 입력에서 입출력 shape, 값 범위, 역전파를 확인합니다. 이는 학습 성능 평가는 아닙니다.


In [ ]:
from src.models.lite_dpas import LiteDPASCsiNetV5, count_parameters
from src.models.csinet import CsiNet

model = LiteDPASCsiNetV5(encoded_dim=ENCODED_DIM).to(DEVICE)
sample = torch.rand(2, 2, 32, 32, device=DEVICE)
output = model(sample)
assert output.shape == sample.shape
assert torch.isfinite(output).all()
assert output.min() >= 0 and output.max() <= 1
((output - sample) ** 2).mean().backward()

baseline_params = count_parameters(CsiNet(encoded_dim=ENCODED_DIM))
lite_params = count_parameters(model)
print('Forward/backward: OK')
print('Input/output:', tuple(sample.shape), tuple(output.shape))
print(f'Parameters: Lite-DPAS {lite_params:,} | CsiNet port {baseline_params:,}')
print(f'Parameter change: {(1 - lite_params / baseline_params) * 100:.1f}% fewer than this CsiNet port')

## 2. 구조 변경 학습

처음에는 1 epoch로 실행 경로를 확인하고, 본 학습에서는 `EPOCHS=1000`으로 설정하세요. V4 checkpoint와 구조가 달라 이어 학습하지 않습니다. 재시작은 V5의 100 epoch periodic checkpoint만 사용하세요.


In [ ]:
from src.train_baseline import train_model

if RUN_TRAINING:
    CHECKPOINT = train_model(
        'litedpas_v5', PROJECT_ROOT, DATA_DIR,
        epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, encoded_dim=ENCODED_DIM,
        run_tag=RUN_TAG, save_every=SAVE_EVERY, resume_from=RESUME_FROM,
        loss_mode=LOSS_MODE, scheduler_patience=SCHEDULER_PATIENCE,
    )
    print('Best validation checkpoint:', CHECKPOINT)
else:
    CHECKPOINT = PROJECT_ROOT / 'runs' / f'litedpas_v5_indoor_cr{COMPRESSION_RATIO}_{EPOCHS}ep_{RUN_TAG}' / 'best.pt'
    print('학습은 실행하지 않았습니다. 설정 셀에서 RUN_TRAINING=True로 바꾸세요.')
    print('예상 checkpoint:', CHECKPOINT)


## 3. Validation NMSE와 복잡도 프로파일

학습 best checkpoint를 불러와 validation 부분집합에서 NMSE, 파라미터, MAC 근사값, batch-1 latency를 기록합니다. MAC 계산은 FFT 연산을 포함하지 않으므로 실제 latency도 함께 비교하세요.


In [ ]:
from src.cost2100 import COST2100Dataset
from src.metrics import sample_nmse_linear
from src.profile_lite_dpas import approximate_macs, measure_latency_ms

if not CHECKPOINT.is_file():
    print('Checkpoint가 없습니다. 먼저 학습 셀을 실행하세요.')
else:
    saved = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    profiled_model = LiteDPASCsiNetV5(encoded_dim=int(saved.get('encoded_dim', ENCODED_DIM)))
    profiled_model.load_state_dict(saved['model'])
    profiled_model = profiled_model.to(DEVICE).eval()
    validation = COST2100Dataset(DATA_DIR / 'DATA_Hvalin.mat')
    n = min(PROFILE_SAMPLES, len(validation))
    val_loader = DataLoader(Subset(validation, range(n)), batch_size=64, shuffle=False, num_workers=0)
    ratios = []
    with torch.inference_mode():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            ratios.append(sample_nmse_linear(batch, profiled_model(batch)).cpu())
    val_nmse_db = float(10 * np.log10(max(torch.cat(ratios).mean().item(), 1e-12)))
    one = torch.zeros(1, 2, 32, 32, device=DEVICE)
    latency = measure_latency_ms(profiled_model, one, warmup=20, repeats=100)
    macs = approximate_macs(profiled_model, one)
    report = {
        'model': 'Lite-DPAS-V5', 'compression_ratio': f"1/{2048 // int(saved.get('encoded_dim', ENCODED_DIM))}",
        'validation_samples': n, 'validation_nmse_db': val_nmse_db,
        'parameters': count_parameters(profiled_model),
        'approx_conv_linear_macs_batch1': macs,
        'batch1_latency_ms': latency, 'device': str(DEVICE), 'test_split_used': False,
        'mac_note': 'FFT, activation, normalization and interpolation excluded',
    }
    display(report)
    report_path = CHECKPOINT.parent / 'profile_validation.json'
    report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
    print('Saved:', report_path)

## 비교 방법

1. V3/V4와 같은 COST2100 indoor train/validation 파일, seed, batch size, feedback dimension, 평가 함수를 사용합니다.
2. 새 구조이므로 `RESUME_FROM=None`, 새 `RUN_TAG`로 학습합니다.
3. 100 epoch마다 저장되는 `checkpoint_epoch_XXXX.pt`에서만 이어 학습합니다.
4. V5에서 NMSE가 개선되지 않으면 다음에는 kernel 크기를 9/15로 ablation하고, 성능과 latency를 함께 비교합니다.
5. 최종 성능은 validation이 아니라 별도 test 평가로 보고하고, test 결과는 모델 선택에 쓰지 않습니다.
